# Capstone (No Key) - Small Business Finance Assistant (Start)
**Day 3 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/capstone_nokey/start/capstone_nokey.ipynb)

**Goal:** Build a retrieval-augmented assistant that answers small-business
finance questions over a mock QuickBooks and TurboTax help knowledge base,
grounded in the retrieved help articles. Everyone finishes the base assistant;
then pick up the tiered stretch goals as time allows.

> **How this runs without an API key.** This is the no-key version of the capstone. Retrieval and fine-tuning already run offline. For generation and the judge it swaps the hosted LLM for a small local model, `google/flan-t5-base`, that downloads once (about 1 GB) and runs on CPU. The agentic stretch replaces the LLM planner with a scripted planner so the tool loop stays visible and deterministic. Everything else is identical to the API capstone; only the call sites change (`generate(...)` and a scripted planner instead of `get_completion(...)` and a `LiteLLMModel` agent).

All data in `capstone/data/` is synthetic mock data. Look for the `# TODO`
markers and fill them in. The `solution/` notebook has the full answer. No API
key needed. Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this capstone (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers faiss-cpu transformers torch peft accelerate

## Setup: data path and a local model
This finds the `capstone/data/` folder locally, or clones the companion repo
when running on Colab, then loads a small local instruction model for
generation. No provider key and no shared client. You do not need to change
this cell.

In [ ]:
import os
import subprocess
import sys

import torch  # noqa: F401  (import torch before faiss to avoid an OpenMP clash on macOS)


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "capstone", "data")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the data files.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
DATA_DIR = os.path.join(root, "capstone", "data")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/flan-t5-base"  # use "google/flan-t5-small" for a smaller download
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
llm.eval()


def generate(prompt, max_new_tokens=96):
    """Greedy text-in, text-out call to the local model (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = llm.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print("Loaded", MODEL_NAME, "and data at", DATA_DIR)

## 1. Load the knowledge base
Read every Markdown help article from `capstone/data/`. Each file is one
source document the assistant is allowed to use.

In [ ]:
import glob

documents = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.md"))):
    with open(path, encoding="utf-8") as f:
        documents.append({"source": os.path.basename(path), "text": f.read()})
print(f"Loaded {len(documents)} help articles:",
      ", ".join(d["source"] for d in documents))

## 2. Chunk, embed, and index
Split each article into overlapping chunks, embed them with a small local
model, and index the vectors with FAISS. Normalized vectors make inner-product
search equal to cosine similarity.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


def chunk_text(text, chunk_size=500, overlap=60):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def build_index(docs, model, chunk_size=500, overlap=60):
    chunks = []
    for doc in docs:
        for piece in chunk_text(doc["text"], chunk_size, overlap):
            chunks.append({"source": doc["source"], "text": piece})
    # TODO: Embed every chunk text with model.encode(..., normalize_embeddings=True),
    # convert to a float32 numpy array, build a faiss.IndexFlatIP of the right
    # dimension, and add the vectors.
    # Hint: vecs = np.array(model.encode([c["text"] for c in chunks],
    #                       normalize_embeddings=True), dtype="float32")
    #       idx = faiss.IndexFlatIP(vecs.shape[1]); idx.add(vecs)
    vecs = ...
    idx = ...
    return chunks, idx


embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunks, index = build_index(documents, embedder)
print(f"Indexed {index.ntotal} chunks.")

## 3. Retrieve and answer, grounded
Retrieve the top chunks for a question, gate on whether they actually answer it,
then answer using only that context and cite the source article. If the context
does not contain the answer, the assistant declines.

In [ ]:
INSTRUCTIONS = (
    "Answer the question using only the context from the QuickBooks and TurboTax "
    "help articles. Keep the answer short."
)


def retrieve(query, model, idx, chunk_list, k=3):
    q = np.array(model.encode([query], normalize_embeddings=True), dtype="float32")
    # TODO: Search the index for the top k chunks and return them with scores.
    # Hint: scores, ids = idx.search(q, k)
    scores, ids = ...
    hits = []
    for score, i in zip(scores[0], ids[0]):
        hit = dict(chunk_list[i])
        hit["score"] = float(score)
        hits.append(hit)
    return hits


def grounded(query, context):
    """Grounding gate: ask the local model whether the context answers the question.

    A hosted model can be told to decline in the answer prompt. A small local model
    answers a plain yes/no question more reliably, so we gate on that first. This
    keeps the assistant from making up an answer when the fact is not in the KB.
    """
    verdict = generate(
        f"Context: {context}\n\nCan the question be answered using only the context "
        f"above? Question: {query}\nAnswer yes or no."
    )
    return verdict.strip().lower().startswith("y")


def rag_answer(query, k=3):
    hits = retrieve(query, embedder, index, chunks, k)
    context = "\n\n".join(h["text"] for h in hits)
    if not grounded(query, context):
        return "I do not have that information in the knowledge base.", hits
    # TODO: Build the answer prompt from INSTRUCTIONS, the context, and the query,
    # call generate(...), and return the answer with a citation of the top source.
    # Hint: answer = generate(f"{INSTRUCTIONS}\nContext: {context}\nQuestion: {query}").strip()
    #       return f"{answer} [{hits[0]['source']}]", hits
    answer = ...
    return f"{answer} [{hits[0]['source']}]", hits


DEMO_QUESTIONS = [
    "What is the default payment term on a QuickBooks invoice?",
    "How much does QuickBooks Payroll Core cost?",
    "Can I deduct a home office, and how much?",
    "Does QuickBooks integrate with Salesforce?",  # not in the KB
]
for q in DEMO_QUESTIONS:
    ans, _ = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}")

## Stretch goals
The base assistant above is what everyone ships. Pick up these in order as time
allows. Hints only here; the full code is in the solution notebook.

1. **Agentic RAG.** Define two tools, `search_help(query)` (wrap `retrieve`) and
   `invoice_total(subtotal, state_code)` that adds sales tax (CA 7.25, NY 4.0,
   TX 6.25, WA 6.5, FL 6.0 percent). Run a hand-written tool loop with a scripted
   `plan_next_step` that searches then computes a 1200 dollar California invoice.
2. **LLM-as-a-judge.** Load `data/golden_qa.json` and, for each pair, ask the
   local model yes or no whether the RAG answer agrees with the expected answer.
   Report the score.
3. **Compare chunking.** Rebuild the index with `chunk_size=200` and compare
   which source article each question retrieves against the default chunks.
4. **LoRA ticket router.** Load `data/support_tickets.json` and adapt Lab 3.2:
   put a LoRA adapter on `distilbert-base-uncased`
   (`AutoModelForSequenceClassification`, `target_modules=["q_lin", "v_lin"]`,
   `task_type="SEQ_CLS"`), train on the 16 example tickets, and route the four
   test tickets into billing, payroll, tax_filing, or integrations.

In [ ]:
# TODO (stretch): your experiments here.